# System 1 caption batch

Captions the middle keyframe of every shot in the worker packages of one batch
(ADR 0025) and uploads one caption package per video to
`{OUTPUT_REPO}/{RELEASE_ID}/captions/artifacts/{BATCH_ID}/`. A restarted session
skips videos that already have a caption package. Run one `BATCH_ID` per session.

Design: `docs/architecture/system1-target.md`, section "Caption Pass". Edit only the settings cell, then Run All.

In [ ]:
# Settings
REPO_URL = "https://github.com/awun0105/Multimodal-Agentic-Retrieval-Engine.git"
REPO_REF = "dev"
RELEASE_ID = "canonical_release_v001"
BATCH_ID = "batch_000"
WORKER_ID = "worker_000"
SOURCE_REPO = "1thesudden/AIOU26_release_test"  # repo holding the worker packages
OUTPUT_REPO = "1thesudden/AIOU26_release_test"  # use 1thesudden/AIOU26_release only for production
LIMIT = 5  # for example 5 for a trial run

In [ ]:
# Setup: secrets, repository, and the worker extra. Runs on Kaggle (GPU T4 x2) or Colab.
import os, subprocess, sys
from pathlib import Path

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

token = secret("HF_TOKEN")
if not token:
    raise RuntimeError("Add an HF_TOKEN secret with write access to the output repo")
os.environ["HF_TOKEN"] = token  # never printed

WORK = Path("/kaggle/temp" if Path("/kaggle").exists() else "/content")
REPO_DIR = WORK / "mare"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "fetch", "--quiet", "origin"], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--quiet", REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR / 'system1'}[worker]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "vietocr==0.3.13"], check=True)
print("commit:", subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True).stdout.strip())

In [ ]:
def system1(*args):
    command = [sys.executable, "-m", "system1.cli", *map(str, args)]
    with subprocess.Popen(command, cwd=REPO_DIR / "system1", stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True) as process:
        for line in process.stdout:
            print(line, end="")
    if process.returncode:
        raise RuntimeError(f"system1 exited with {process.returncode}")

args = ["caption-batch", "--batch-id", BATCH_ID, "--worker-id", WORKER_ID, "--release-id", RELEASE_ID,
        "--source-repo", SOURCE_REPO, "--output-repo", OUTPUT_REPO, "--work-root", WORK / "caption_runtime",
        "--persist-dir", Path("/kaggle/working" if Path("/kaggle").exists() else "/content") / "caption_report"]
if LIMIT:
    args += ["--limit", LIMIT]
system1(*args)